# Step 2: Launch AutoGluon Multimodal Training on SageMaker

Uses `ModelTrainer` (SDK v3) with the AWS-managed AutoGluon DLC image.

Requires a GPU instance (ml.g4dn.xlarge) for the text transformer.

## Configuration

In [ ]:
import boto3
import sagemaker

REGION = boto3.session.Session().region_name
sess = sagemaker.session.Session()
BUCKET = sess.default_bucket()
S3_PREFIX = "autogluon-multimodal"

S3_TRAIN = f"s3://{BUCKET}/{S3_PREFIX}/processed/train/"
S3_VALIDATION = f"s3://{BUCKET}/{S3_PREFIX}/processed/validation/"
S3_OUTPUT = f"s3://{BUCKET}/{S3_PREFIX}/model/"
INSTANCE_TYPE = "ml.g4dn.xlarge"
AG_VERSION = "1.5"
PY_VERSION = "py312"
JOB_NAME = "autogluon-multimodal"

print(f"Region:       {REGION}")
print(f"S3 train:     {S3_TRAIN}")
print(f"S3 val:       {S3_VALIDATION}")
print(f"S3 output:    {S3_OUTPUT}")

## Discover IAM Role

In [ ]:
iam = boto3.client("iam")
role_arn = None
paginator = iam.get_paginator("list_roles")
for page in paginator.paginate():
    for role in page["Roles"]:
        if "SageMaker" in role["RoleName"] or "sagemaker" in role["RoleName"]:
            role_arn = role["Arn"]
            break
    if role_arn:
        break

if not role_arn:
    raise ValueError("No SageMaker IAM role found. Set role_arn manually.")

print(f"Role: {role_arn}")

## Imports and Training Image

In [ ]:
from sagemaker.train import ModelTrainer
from sagemaker.core.helper.session_helper import Session
from sagemaker.core import image_uris
from sagemaker.core.training.configs import (
    Compute,
    SourceCode,
    OutputDataConfig,
    StoppingCondition,
)

session = Session()

image_uri = image_uris.retrieve(
    "autogluon",
    region=REGION,
    version=AG_VERSION,
    py_version=PY_VERSION,
    image_scope="training",
    instance_type=INSTANCE_TYPE,
)
print(f"Image URI: {image_uri}")

## Define Hyperparameters

In [ ]:
hyperparameters = {
    "numerical-feature-names": "CustServ Calls,Account Length",
    "categorical-feature-names": "plan,limit",
    "textual-feature-names": "text",
    "label-name": "y",
    "problem_type": "classification",
    "eval_metric": "roc_auc",
    "presets": "medium_quality",
    "pretrained-transformer": "google/electra-small-discriminator",
    "verbosity": 2,
}

## Create ModelTrainer and Launch Training

In [ ]:
trainer = ModelTrainer(
    training_image=image_uri,
    role=role_arn,
    source_code=SourceCode(
        source_dir=".",
        entry_script="train.py",
    ),
    compute=Compute(
        instance_type=INSTANCE_TYPE,
        instance_count=1,
        volume_size_in_gb=50,
        keep_alive_period_in_seconds=0,
    ),
    output_data_config=OutputDataConfig(
        s3_output_path=S3_OUTPUT,
    ),
    hyperparameters=hyperparameters,
    base_job_name=JOB_NAME,
    stopping_condition=StoppingCondition(max_runtime_in_seconds=14400),
    sagemaker_session=session,
)

training_job = trainer.train(
    input_data_config=[
        {
            "channel_name": "train",
            "data_source": {
                "s3_data_source": {
                    "s3_uri": S3_TRAIN,
                    "s3_data_type": "S3Prefix",
                }
            },
        },
        {
            "channel_name": "validation",
            "data_source": {
                "s3_data_source": {
                    "s3_uri": S3_VALIDATION,
                    "s3_data_type": "S3Prefix",
                }
            },
        },
    ],
    wait=True,
    logs=True,
)

## Training Results

In [ ]:
job_name = training_job.name
print(f"Training complete: {job_name}")
print(f"Console: https://{REGION}.console.aws.amazon.com/sagemaker/home?region={REGION}#/jobs/{job_name}")